# Phase 1: EDA

Answers the 5 Phase 1 questions from the project plan. Heavy lifting lives in `src/rogii_wellbore/`; this notebook drives, visualizes, and records findings.

**Questions:**
1. Eval-zone size per well — distribution of `TVT_input.isna()` count and span
2. GR comparability across wells — per-well GR distributions; per-well normalization needed?
3. Typewell TVT coverage — does typewell cover lateral well TVT range?
4. MD step uniformity — within-well and across-well
5. Train/test well overlap — already established (3 test wells = first 3 train wells)

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from rogii_wellbore import data
from rogii_wellbore.config import RAW_DIR

print("python :", sys.executable)
print("raw_dir:", RAW_DIR)

In [ ]:
hw_train = data.load_horizontal("train")
print(hw_train.shape)
hw_train.head(2)

In [ ]:
def well_eval_stats(g: pd.DataFrame) -> pd.Series:
    n = len(g)
    mask = g["TVT_input"].isna()
    n_eval = int(mask.sum())
    if n_eval == 0:
        return pd.Series(
            {
                "n_rows": n,
                "n_eval": 0,
                "n_known": n,
                "eval_frac": 0.0,
                "eval_start": np.nan,
                "eval_end": np.nan,
                "eval_contiguous": True,
            }
        )
    eval_idx = np.where(mask.values)[0]
    start, end = int(eval_idx[0]), int(eval_idx[-1])
    contiguous = (end - start + 1) == n_eval
    return pd.Series(
        {
            "n_rows": n,
            "n_eval": n_eval,
            "n_known": n - n_eval,
            "eval_frac": n_eval / n,
            "eval_start": start,
            "eval_end": end,
            "eval_contiguous": contiguous,
        }
    )


eval_stats = hw_train.groupby("well_id").apply(well_eval_stats, include_groups=False)
eval_stats.head()

In [ ]:
print(f"wells: {len(eval_stats)}")
print(f"all contiguous tail eval zones: {eval_stats['eval_contiguous'].all()}")
print(f"any well with 0 eval rows: {(eval_stats['n_eval'] == 0).sum()}")
print(f"any well with 0 known rows: {(eval_stats['n_known'] == 0).sum()}")
print()
print("summary stats:")
eval_stats[["n_rows", "n_eval", "n_known", "eval_frac"]].describe(
    percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
)

In [ ]:
ends_at_last = eval_stats["eval_end"] == eval_stats["n_rows"] - 1
print(f"eval zone ends at last row: {ends_at_last.sum()} / {len(eval_stats)}")
if not ends_at_last.all():
    print("\nwells where eval zone does NOT end at last row:")
    display(eval_stats[~ends_at_last].head(10))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(eval_stats["n_rows"], bins=50)
axes[0].set_xlabel("rows per well")
axes[0].set_ylabel("count")
axes[0].set_title("Well length")

axes[1].hist(eval_stats["n_eval"], bins=50)
axes[1].set_xlabel("eval rows per well")
axes[1].set_title("Eval zone size (rows)")

axes[2].hist(eval_stats["eval_frac"], bins=50)
axes[2].set_xlabel("eval rows / total rows")
axes[2].set_title("Eval zone fraction")

plt.tight_layout()
plt.savefig("../reports/figures/01_eval_zone_distributions.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
eval_stats.to_parquet("../data/interim/eval_zone_stats.parquet")
print("saved:", "data/interim/eval_zone_stats.parquet", eval_stats.shape)

In [ ]:
Path("docs").mkdir(exist_ok=True)
Path("../docs/01_eda_notes.md").write_text("""# Phase 1: EDA notes

## Q1. Eval-zone size per well

**All 773 train wells have the same eval-zone structure: a contiguous block of `TVT_input.isna()` rows at the tail of the well.** The mask never appears mid-well; it always extends from some row `k` to the final row.

This is forward extrapolation along the lateral, not random infilling. Implication for Phase 2: causal/autoregressive models are natural; no future leak during training.

### Distribution (773 wells)

| stat              | total rows | known rows | eval rows | eval fraction |
|-------------------|-----------:|-----------:|----------:|--------------:|
| min               |      2,058 |        851 |       407 |         0.198 |
| 5th pct           |      4,652 |      1,346 |     2,947 |         0.625 |
| median            |      6,576 |      1,703 |     4,840 |         0.740 |
| mean              |      6,588 |      1,692 |     4,895 |         0.733 |
| 95th pct          |      8,614 |      2,053 |     6,918 |         0.819 |
| max               |     12,141 |      2,392 |    10,052 |         0.875 |

### Observations
- **Known segment is tight** (851-2392 rows, std 217). Each well gives the model roughly the same amount of context.
- **Eval segment is wide** (407-10052 rows, std 1301). 25x range. A model needs to handle both short and very long forwad predictions.
- **One low-mask outlier** (eval_frac ≈ 0.20) — flagged but not investigated.
- Submission `id` integer = horizontal_well CSV row index; eval rows are exactly the rows where `TVT_input` is NaN.

### Figure
`reports/figures/01_eval_zone_distributions.png`
""")
print("wrote docs/01_eda_notes.md")

In [ ]:
gr_stats = hw_train.groupby("well_id")["GR"].agg(
    [
        "count",
        "mean",
        "std",
        "min",
        "max",
        lambda s: s.quantile(0.05),
        lambda s: s.quantile(0.50),
        lambda s: s.quantile(0.95),
    ]
)
gr_stats.columns = ["n", "mean", "std", "min", "max", "p05", "p50", "p95"]
print("NaN GR rows:", hw_train["GR"].isna().sum())
gr_stats.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(gr_stats["mean"], bins=50)
axes[0].set_xlabel("per-well mean GR")
axes[0].set_ylabel("count")
axes[0].set_title("Distribution of per-well GR means")

axes[1].hist(gr_stats["std"], bins=50)
axes[1].set_xlabel("per-well std GR")
axes[1].set_title("Distribution of per-well GR stds")

# Per-well range (p95 - p05) gives a robust within-well spread
axes[2].scatter(gr_stats["mean"], gr_stats["p95"] - gr_stats["p05"], s=4, alpha=0.4)
axes[2].set_xlabel("per-well mean GR")
axes[2].set_ylabel("per-well p95 - p05 GR")
axes[2].set_title("Mean vs spread (per well)")

plt.tight_layout()
plt.savefig("../reports/figures/02_gr_per_well_stats.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
across_well_mean_std = gr_stats["mean"].std()  # how much per-well means vary
within_well_std_med = gr_stats["std"].median()  # typical per-well variation
ratio = across_well_mean_std / within_well_std_med

print(f"std of per-well means:    {across_well_mean_std:.2f}")
print(f"median of per-well stds:  {within_well_std_med:.2f}")
print(f"ratio (across / within):  {ratio:.2f}")
print()
print("Interpretation:")
print("  ratio << 1 -> wells are comparable as-is; per-well norm probably unnecessary")
print("  ratio ~ 1  -> per-well norm is sensible")
print("  ratio >> 1 -> per-well norm essential; or features become well-id-dependent")

In [ ]:
hw_train["gr_nan"] = hw_train["GR"].isna()
hw_train["in_eval"] = hw_train["TVT_input"].isna()

total = len(hw_train)
print(f"total rows:               {total:,}")
print(f"GR NaN:                   {hw_train['gr_nan'].sum():,} ({hw_train['gr_nan'].mean():.1%})")
print(f"GR NaN AND in eval zone:  {(hw_train['gr_nan'] & hw_train['in_eval']).sum():,}")
print(f"GR NaN AND in known zone: {(hw_train['gr_nan'] & ~hw_train['in_eval']).sum():,}")
print()
# Conditional rates
eval_rows = hw_train["in_eval"].sum()
known_rows = total - eval_rows
print(
    f"P(GR NaN | in eval zone):  {(hw_train['gr_nan'] & hw_train['in_eval']).sum() / eval_rows:.1%}"
)
print(
    f"P(GR NaN | in known zone): {(hw_train['gr_nan'] & ~hw_train['in_eval']).sum() / known_rows:.1%}"
)

# how many wells have ANY GR available in the eval zone?
gr_in_eval = hw_train[hw_train["in_eval"]].groupby("well_id")["GR"].apply(lambda s: s.notna().sum())
print(f"\nwells with >=1 GR value in eval zone: {(gr_in_eval > 0).sum()} / {len(gr_in_eval)}")
print(f"wells with ZERO GR in eval zone:      {(gr_in_eval == 0).sum()}")
gr_in_eval.describe()

In [ ]:
g = hw_train[hw_train["well_id"] == "000d7d20"].copy().reset_index(drop=True)
nan_mask = g["GR"].isna().values
# Run-length encoding of NaN runs
changes = np.diff(np.concatenate(([0], nan_mask.astype(int), [0])))
starts = np.where(changes == 1)[0]
ends = np.where(changes == -1)[0]
run_lengths = ends - starts
print(f"well 000d7d20: {nan_mask.sum()} / {len(nan_mask)} GR NaN ({nan_mask.mean():.1%})")
print(f"number of NaN runs: {len(run_lengths)}")
print("run-length distribution:")
print(pd.Series(run_lengths).describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))
print(f"\nfirst 20 run lengths: {run_lengths[:20].tolist()}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(g["row_idx"], g["GR"], lw=0.4)
ax.axvline(
    g[g["TVT_input"].isna()]["row_idx"].iloc[0], color="red", ls="--", lw=1, label="eval start"
)
ax.set_xlabel("row_idx")
ax.set_ylabel("GR")
ax.set_title("GR vs row_idx, well 000d7d20")
ax.legend()
plt.tight_layout()
plt.savefig("../reports/figures/02b_gr_one_well.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
gr_nan_rate = hw_train.groupby("well_id")["GR"].apply(lambda s: s.isna().mean())
print("Per-well GR NaN rate distribution:")
print(gr_nan_rate.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))
print(f"\nwells with > 50% GR NaN: {(gr_nan_rate > 0.5).sum()}")
print(f"wells with > 70% GR NaN: {(gr_nan_rate > 0.7).sum()}")

In [ ]:
worst = gr_nan_rate.sort_values(ascending=False).head(10)
print("10 worst wells by GR NaN rate:")
print(worst)

# save per-well GR stats for later phases
gr_stats["nan_rate"] = gr_nan_rate
gr_stats.to_parquet("../data/interim/gr_per_well_stats.parquet")
print(f"\nsaved: data/interim/gr_per_well_stats.parquet  {gr_stats.shape}")

In [ ]:
q2 = """

## Q2. GR comparability across wells

**Per-well normalization is sensible (and probably necessary).** The ratio of across-well variation to within-well variation is ≈ 0.89 — wells differ in mean GR about as much as a single well varies internally. A raw `GR = 80` means different things in different wells.

### Per-well summary (773 wells)

| stat            | mean GR | std GR | nan rate |
|-----------------|--------:|-------:|---------:|
| min             |    37.2 |    8.3 |    0.7%  |
| 5th pct         |    65.4 |   12.3 |    3.8%  |
| median          |    86.6 |   17.3 |   27.7%  |
| 95th pct        |   118.2 |   25.3 |   60.3%  |
| max             |   130.5 |   35.1 |   80.1%  |

- Across-well std of per-well means: 15.4
- Median of per-well stds: 17.3
- Ratio (across / within): **0.89** → per-well z-score is the right default

### Bimodality
The distribution of per-well GR means is bimodal (peaks ~85 and ~115), suggesting two clusters of wells (likely different formations, fields, or tool calibrations). A naive per-well z-score erases this cluster signal. Worth testing typewell-conditioned normalization in Phase 2 as an alternative.

### Missing GR
- **29.6% of GR values are NaN overall.** Rate is similar in known (23.5%) and eval (31.7%) zones — GR is *not* cut off at the eval boundary; it's intermittent tool dropouts.
- Per-well NaN rate varies widely: 1% (best) to 80% (worst), median 28%, std 19%.
- **145 wells (19%) have > 50% NaN**, 4 wells > 70%. Phase 2 should evaluate whether these are usable, downweight them, or drop them.
- NaN runs are short (median 1, 99th pct 11, max 19 rows on the spot-check well). Linear interpolation per well is safe; cap on max run-length to be confirmed across all wells.

### Outliers
Spot-checked well `000d7d20` shows a single-row GR spike to ~210 around row 4400 (vs ~95 baseline). Likely sensor glitch. Phase 2 needs robust scaling or outlier clipping.

### Figures
- `reports/figures/02_gr_per_well_stats.png` — distributions of per-well mean/std/spread
- `reports/figures/02b_gr_one_well.png` — GR trace for well 000d7d20 (gaps + outlier visible)
"""
Path("../docs/01_eda_notes.md").write_text(Path("../docs/01_eda_notes.md").read_text() + q2)
print("appended Q2 to docs/01_eda_notes.md")